In [29]:
import polars as pl
import altair as alt
import functions
import importlib

In [55]:
importlib.reload(functions)
# Generate dataset using the function with treatment impact
df = functions.generate_dataset(
    num_units=100,
    num_obs=10000,
    impact=0.0,
    distr="geom",
    beta_size=100,
    baseline_conversion_rate=0.2,
)

In [56]:
unit_summary = df.group_by("unit_id").agg(
    [
        pl.len().alias("num_observations"),
        pl.first("base_success_rate").alias("base_success_rate"),
        pl.mean("outcome").alias("observed_success_rate"),        
    ]
).with_columns(
    (pl.col("num_observations") / pl.col('num_observations').sum()).alias("percent_of_total")
)

# Sort unit_summary by num_observations descending
unit_summary_sorted = unit_summary.sort("num_observations", descending=True)

base_chart = (
    alt.Chart(unit_summary_sorted).encode(
        alt.X("unit_id:O", sort=None, title="Unit ID"),
    ))

chart = (
    base_chart.mark_bar().encode(        
        alt.Y("percent_of_total:Q", title="% of Total Observations"),        
    ) + 
    base_chart.mark_line().encode(        
        alt.Y("observed_success_rate:Q", title="% CVR"),        
    )
).properties(
        title="Percent of Total Observations per Unit (Sorted)", width=800, height=400
    )

chart

alt.LayerChart(...)

In [27]:
from tqdm.notebook import tqdm
import iw

estimates = []
for i in tqdm(range(500)):
    df = functions.generate_dataset(
        num_units=100,
        num_obs=3000,
        impact=0.0,
        distr="geom",
        beta_size=100,
        baseline_conversion_rate=0.2,
    )
    # df = iw.generate_experiment_dataframe(
    #     sessions_skew=sessions_skew,
    #     num_users=60,
    #     baseline_conversion_rate=baseline_conversion_rate,
    #     cvr_decay_factor=cvr_decay_factor,
    #     beta_size=beta_size,
    # )
    r = functions.analyze_treatment_effect(df)
    [d.update({"i": i}) for d in r]
    estimates.extend(r)

  0%|          | 0/500 [00:00<?, ?it/s]

In [28]:
pl.DataFrame(estimates).group_by("model").agg(
    [
        pl.mean("estimate").alias("mean_estimate"),
        pl.mean("significant").alias("power"),
        # confidence interval width
        (pl.col("ci_upper") - pl.col("ci_lower")).mean().alias("mean_ci_width"),
        (pl.col("ci_upper") - pl.col("ci_lower")).median().alias("median_ci_width"),
    ]
)

model,mean_estimate,power,mean_ci_width,median_ci_width
str,f64,f64,f64,f64
"""clustered_ols""",-0.001533,0.064,0.073519,0.073208
"""bootstrap_optimized""",-0.001555,0.066,0.073257,0.073217
"""unit_level_ols_unweighted""",-0.002216,0.052,0.114525,0.11329
"""obs_level_randomization""",0.000372,0.034,0.060357,0.060307
"""delta_method""",-0.001533,0.064,0.073897,0.073607
